# Module 8C: DQ Dashboard - Streamlit in Snowflake

## Learning Objectives
- Build and deploy a **Streamlit in Snowflake (SiS)** DQ monitoring app
- Provide self-service DQ visibility to business users

> **Business Value:** Business users get a polished, interactive dashboard inside Snowflake. No Power BI license, no VPN, no external hosting -- just Snowflake credentials and a browser.

---
> **Role:** `CORP_DQ_ADMIN` | **Time:** ~45 minutes | **Variant:** Streamlit in Snowflake
> **Reference:** [STUDENT_GUIDE.md](../guide/STUDENT_GUIDE.md) — Module 8 compares the 3 dashboard variants (Native, Python, Streamlit).


> **What this does:** Sets your session context to the lab role, database, and warehouse.


In [ ]:
USE ROLE CORP_DQ_ADMIN;
USE DATABASE CORP_DWH;
USE WAREHOUSE DQ_LAB_WH;

---
## Shared Setup: Create DQ Reporting Views

> **Note:** These views are identical across all Module 8 variants (8A/8B/8C). If you already ran another variant, these views already exist -- running them again is safe (`CREATE OR REPLACE`).

> **Business Value:** BI tools cannot call table functions directly. Views provide a stable, queryable interface.

In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_RESULTS_FLAT AS
SELECT
    r.REF_ENTITY_NAME AS TABLE_NAME, r.METRIC_NAME,
    r.ARGUMENT_NAMES AS COLUMN_CHECKED, r.VALUE AS METRIC_VALUE,
    r.EXPECTATION_NAME, r.EXPECTATION_RESULT, r.MEASUREMENT_TIME,
    COALESCE(c.SEVERITY, 'MEDIUM') AS SEVERITY,
    COALESCE(c.OWNER, 'Unassigned') AS RULE_OWNER,
    COALESCE(c.RULE_TYPE, 'SYSTEM') AS RULE_TYPE,
    CASE WHEN r.EXPECTATION_RESULT = 'MET' THEN 'PASS'
         WHEN r.EXPECTATION_RESULT = 'NOT_MET' THEN 'FAIL'
         ELSE 'NO_EXPECTATION' END AS STATUS
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'
)) r
LEFT JOIN CORP_DWH.DQ.RULES_CATALOG c
    ON UPPER(r.METRIC_NAME) LIKE '%' || REPLACE(UPPER(c.RULE_NAME), ' ', '_') || '%';

> **What this does:** Creates V_DQ_SCORECARD view that calculates per-table health scores from the latest DQ expectation results.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_SCORECARD AS
WITH latest AS (
    SELECT 'CORP_DWH.GOLD.DIM_CUSTOMER' AS TABLE_NAME,
        METRIC_NAME, ARGUMENT_NAMES, VALUE, EXPECTATION_NAME, EXPECTATION_RESULT, MEASUREMENT_TIME,
        ROW_NUMBER() OVER (PARTITION BY METRIC_NAME, ARGUMENT_NAMES ORDER BY MEASUREMENT_TIME DESC) AS RN
    FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
        REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
    WHERE EXPECTATION_NAME IS NOT NULL
)
SELECT TABLE_NAME,
    COUNT(*) AS TOTAL_EXPECTATIONS,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) AS PASSED,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'NOT_MET' THEN 1 END) AS FAILED,
    ROUND(100.0 * COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) / NULLIF(COUNT(*), 0), 1) AS HEALTH_SCORE_PCT,
    MAX(MEASUREMENT_TIME) AS LAST_EVALUATED
FROM latest WHERE RN = 1 GROUP BY TABLE_NAME;

> **What this does:** Creates V_DQ_TREND view that provides hourly time-series data for charting quality metrics over time.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_TREND AS
SELECT DATE_TRUNC('HOUR', MEASUREMENT_TIME) AS MEASUREMENT_HOUR,
    METRIC_NAME, VALUE AS METRIC_VALUE, EXPECTATION_RESULT, MEASUREMENT_TIME
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
WHERE EXPECTATION_NAME IS NOT NULL ORDER BY MEASUREMENT_TIME DESC;

> **What this does:** Creates V_DQ_EXECUTIVE_SUMMARY view that aggregates all checks into a single overall health percentage.


In [ ]:
CREATE OR REPLACE VIEW CORP_DWH.DQ.V_DQ_EXECUTIVE_SUMMARY AS
SELECT 'CORP_DWH' AS DATA_ESTATE, COUNT(*) AS TOTAL_CHECKS,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) AS CHECKS_PASSING,
    COUNT(CASE WHEN EXPECTATION_RESULT = 'NOT_MET' THEN 1 END) AS CHECKS_FAILING,
    ROUND(100.0 * COUNT(CASE WHEN EXPECTATION_RESULT = 'MET' THEN 1 END) / NULLIF(COUNT(*), 0), 1) AS OVERALL_HEALTH_PCT,
    CURRENT_TIMESTAMP() AS AS_OF
FROM TABLE(SNOWFLAKE.LOCAL.DATA_QUALITY_MONITORING_RESULTS(
    REF_ENTITY_NAME => 'CORP_DWH.GOLD.DIM_CUSTOMER', REF_ENTITY_DOMAIN => 'TABLE'))
WHERE EXPECTATION_NAME IS NOT NULL;

---
## Create the App Schema and Stage

> **What this does:** Creates the APPS schema and a stage to host the Streamlit app files.


In [ ]:
CREATE SCHEMA IF NOT EXISTS CORP_DWH.APPS COMMENT = 'Streamlit applications';
CREATE OR REPLACE STAGE CORP_DWH.APPS.DQ_DASHBOARD_STAGE DIRECTORY = (ENABLE = TRUE);

---
## The Streamlit App Code

Below is the complete app. Review it, then we deploy it to Snowflake.

---
## App Features

The Streamlit app (`streamlit_dq_app.py`) provides 5 interactive tabs:

| Tab | Content |
|-----|--------|
| **Overview** | Pass/Fail charts, severity breakdown, top violations, health by table |
| **Trend** | Line charts showing quality over time (per hour + per metric) |
| **Failures** | Filterable table of all failing rules with severity filter |
| **Rules Catalog** | Provisioned vs pending rules, owner assignments |
| **Drill-Down** | Investigate specific issues: NULLs, duplicates, orphans, staleness |

> **Source code:** See `notebooks/streamlit_dq_app.py` in the repo for the full implementation.


---
## Deploy the App

> **What this does:** Uploads the Streamlit app code to the stage and creates the Streamlit object in Snowflake for deployment.


In [ ]:
from snowflake.snowpark.context import get_active_session
import tempfile, os
session = get_active_session()

# The full app code (same as streamlit_dq_app.py in the repo)
app_code = """ + repr(app_code) + """

# Write to a temp file with a clean name
tmp_dir = tempfile.mkdtemp()
app_path = os.path.join(tmp_dir, "app.py")
with open(app_path, 'w') as f:
    f.write(app_code)

try:
    session.file.put(app_path, "@CORP_DWH.APPS.DQ_DASHBOARD_STAGE/", auto_compress=False, overwrite=True)
    print("[OK] Uploaded app.py to stage")
    
    session.sql("""
        CREATE OR REPLACE STREAMLIT CORP_DWH.APPS.DQ_DASHBOARD
            ROOT_LOCATION = '@CORP_DWH.APPS.DQ_DASHBOARD_STAGE'
            MAIN_FILE = 'app.py'
            QUERY_WAREHOUSE = DQ_LAB_WH
            COMMENT = 'Data Quality Monitoring Dashboard - DQ HOL Module 8C'
    """).collect()
    print("[OK] Streamlit app created: CORP_DWH.APPS.DQ_DASHBOARD")
    print("\nAccess: Snowsight > Projects > Streamlit > DQ_DASHBOARD")
    
    # Grant access
    session.sql("GRANT USAGE ON STREAMLIT CORP_DWH.APPS.DQ_DASHBOARD TO ROLE CORP_DQ_ADMIN").collect()
    print("[OK] Granted access to CORP_DQ_ADMIN")
except Exception as e:
    print(f"[INFO] {str(e)[:120]}")
    print("\nManual deploy: Go to Projects > Streamlit > + Streamlit App")
    print("Paste the code from streamlit_dq_app.py in the repo.")
finally:
    os.unlink(app_path)
    os.rmdir(tmp_dir)


---
## Grant Access to Business Users

> **What this does:** Grants the DQ steward role access to the deployed Streamlit dashboard.


In [ ]:
GRANT USAGE ON STREAMLIT CORP_DWH.APPS.DQ_DASHBOARD TO ROLE CORP_DQ_STEWARD;

---
## Comparison: When to Use Each Variant

| Variant | Best For | Pros | Cons |
|---------|----------|------|------|
| **8A: Native Dashboards** | Quick SQL monitoring | Zero code, built-in | Limited interactivity |
| **8B: Python Charts** | Data engineers | Rich customization | Not shareable standalone |
| **8C: Streamlit (SiS)** | Business self-service | Interactive, deployed, shareable | Requires app code |
| **Power BI** | Enterprise BI teams | Familiar, RLS, scheduling | External tool, license |

---
## Checkpoint

> **What this does:** Verifies your work so far. All checks should show [PASS].


In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print("=" * 50)
print("CHECKPOINT: Dashboard Views + Streamlit App")
print("=" * 50)

passed = 0
total = 5

# Check 4 dashboard views
for v in ['V_DQ_RESULTS_FLAT', 'V_DQ_SCORECARD', 'V_DQ_TREND', 'V_DQ_EXECUTIVE_SUMMARY']:
    try:
        cnt = session.sql(f"SELECT COUNT(*) AS C FROM CORP_DWH.DQ.{v}").collect()[0]['C']
        print(f"  [PASS] {v} -- {cnt} rows")
        passed += 1
    except Exception as e:
        print(f"  [FAIL] {v}: {str(e)[:50]}")

# Check Streamlit app exists
try:
    result = session.sql("SHOW STREAMLIT LIKE 'DQ_DASHBOARD' IN SCHEMA CORP_DWH.APPS").collect()
    if len(result) > 0:
        print(f"  [PASS] Streamlit app DQ_DASHBOARD deployed")
        passed += 1
    else:
        print(f"  [WAIT] Streamlit app not yet deployed (run the deploy cell above)")
except Exception as e:
    print(f"  [WAIT] Streamlit app not yet deployed: {str(e)[:50]}")

print(f"\nResult: {passed}/{total} checks passed")
print("=" * 50)


---
**Next:** Proceed to `9_TEARDOWN` (optional cleanup).